# 06 Advanced RAG

**RAG (Retrieval-Augmented Generation)** means: search your knowledge base for relevant text, pass that text to the LLM, then generate an answer grounded in those sources.

**Basic RAG** does: chunk documents, embed them, retrieve top matches, answer.

**This notebook** goes further. Each section below adds one technique that fixes a common RAG failure mode: bad chunks, wrong matches, duplicate results, or answers not tied to the question.

Run the cells in order. Watch `DATAFLOW` output to see retrieve, then generate (or agent tool loops).


Set up the project path and reload shared helpers from this repo.


In [9]:
import sys
import importlib
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "shared").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

# Pick up edits to shared/ without a full kernel restart (re-run this cell)
for _name in (
    "shared.dataflow",
    "shared.notebook_display",
    "shared.llm",
    "shared.bootcamp_fixtures",
):
    importlib.reload(importlib.import_module(_name))

print(f"Project root: {ROOT}")


Project root: c:\Users\Azooo\langchain-bootcamp


Check which API keys and provider settings are available in `.env`.


In [10]:
import os

def env_status():
    keys = {
        "DEEPSEEK_API_KEY": bool(os.getenv("DEEPSEEK_API_KEY")),
        "DEEPSEEK_MODEL": os.getenv("DEEPSEEK_MODEL", "deepseek-v4-flash"),
        "LLM_PROVIDER": os.getenv("LLM_PROVIDER", "deepseek"),
        "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
        "LANGSMITH_API_KEY": bool(os.getenv("LANGSMITH_API_KEY")),
    }
    for k, v in keys.items():
        print(f"{k}: {v}")
    return keys

ENV = env_status()


DEEPSEEK_API_KEY: True
DEEPSEEK_MODEL: deepseek-v4-flash
LLM_PROVIDER: deepseek
TAVILY_API_KEY: True
LANGSMITH_API_KEY: True


Load the chat model helper used by live cells below.


In [11]:
import os
from shared.llm import get_model

def require_llm():
    return get_model()


Import DATAFLOW printers once, they label each agent and RAG step so you read a run like a wizard tracing a spell. Later cells call print_agent_dataflow or print_rag_dataflow; watch those blocks closely.


In [12]:
from shared.dataflow import preview, print_agent_dataflow, print_rag_dataflow, print_final_state


Limit PyTorch thread count before embedding work, long Wikipedia pages plus local models can peg CPU. Print confirms the guard ran; adaptive chunking below stays responsive.


In [13]:
import os
# Windows/Jupyter stability for sentence-transformers / PyTorch
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
print("thread guard applied")


thread guard applied


## Adaptive chunking

**What:** Try several split strategies on each document and pick the best one before indexing.

**Problem it fixes:** A single chunk size fails in both directions. Chunks too large bury the answer in noise. Chunks too small break sentences and lose meaning.

**How it works:** Split the same doc with page, sentence, recursive, semantic, and optional `llm_regex` methods. Score each split on size compliance and block integrity. Keep the strategy with the highest weighted score for that file.

**In this cell:** Scores real travel docs in `data/` and prints `BEST:` per source file.


In [14]:
import importlib
from pathlib import Path
import shared.rag_adaptive as rag_adaptive
importlib.reload(rag_adaptive)

print("rag_adaptive:", rag_adaptive.__file__)
from shared.bootcamp_fixtures import load_travel_corpus

docs = load_travel_corpus(ROOT)
print("Sample source:", docs[0].metadata.get("source", "?")[:60])

print("Building chunking strategies (first run may download ~90MB embedding model)...")
strategies = rag_adaptive.build_sync_strategies(include_semantic=True)
strategies = rag_adaptive.finalize_strategies(strategies)
available = [m for m in rag_adaptive.CHUNKING_METHODS if m in strategies]
print("Methods evaluated:", ", ".join(available))
if "llm_regex" not in strategies:
    print("llm_regex skipped, set DEEPSEEK_API_KEY (or OPENAI_API_KEY) to include it")

print("Scoring each document (semantic = local MiniLM embeddings, slow on long Wikipedia pages)...")
for doc in docs:
    src = Path(doc.metadata.get("source", "")).name
    print(f" scoring {src}...", flush=True)
    best, parts, score, table = rag_adaptive.pick_best_strategy(doc.page_content, strategies)
    print(f"\n{src} to BEST: {best} (weighted={score:.2f}, chunks={len(parts)})")
    for method in available:
        row = table.get(method, {})
        if "error" in row:
            print(f" {method:22s} ERROR: {row['error'][:60]}")
        else:
            m = row["metrics"]
            print(
                f" {method:22s} SC={m['size_compliance']:.2f} "
                f"BI={m['block_integrity']:.2f} weighted={row['weighted']:.2f} n={row['n_chunks']}"
            )


rag_adaptive: c:\Users\Azooo\langchain-bootcamp\shared\rag_adaptive.py
Loaded 9 documents from data/
Sample source: c:\Users\Azooo\langchain-bootcamp\data\eu_air_passenger_righ
Building chunking strategies (first run may download ~90MB embedding model)...
Loading semantic chunker model (sentence-transformers/all-MiniLM-L6-v2)...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4577.53it/s]


Semantic chunker ready.
llm_regex enabled via DeepSeek (deepseek-v4-flash)
Methods evaluated: page, sentence, langch_recurs_default, langch_recurs_1100, our_recurs_1100, our_recurs_600, semantic, llm_regex
Scoring each document (semantic = local MiniLM embeddings, slow on long Wikipedia pages)...
 scoring eu_air_passenger_rights.txt...

eu_air_passenger_rights.txt to BEST: langch_recurs_1100 (weighted=1.00, chunks=2)
 page                   SC=0.00 BI=1.00 weighted=0.50 n=2
 sentence               SC=1.00 BI=0.97 weighted=0.98 n=3
 langch_recurs_default  SC=0.67 BI=0.98 weighted=0.82 n=3
 langch_recurs_1100     SC=1.00 BI=1.00 weighted=1.00 n=2
 our_recurs_1100        SC=1.00 BI=1.00 weighted=1.00 n=2
 our_recurs_600         SC=1.00 BI=1.00 weighted=1.00 n=3
 semantic               SC=0.50 BI=1.00 weighted=0.75 n=2
 llm_regex              SC=0.00 BI=1.00 weighted=0.50 n=1
 scoring state_dept_portugal_travel.txt...

state_dept_portugal_travel.txt to BEST: our_recurs_1100 (weighted=1.00,

## Contextual indexing

**What:** Prepend a short header (source file, strategy) to each chunk before embedding and storing in Chroma.

**Problem it fixes:** A bare chunk often looks generic in embedding space. The model cannot tell whether a paragraph came from visa rules, insurance, or Lisbon tourism.

**How it works:** Build `Document: … / Strategy: … / --- / {chunk text}`, embed the full string, persist to `.chroma_advanced_rag`. Search still uses vectors, but each vector carries document context.

**In this cell:** Indexes contextual chunks and prints the Chroma path.


In [15]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
import shared.rag_adaptive as rag_adaptive

CHROMA_DIR = str(ROOT / ".chroma_advanced_rag")
embeddings = rag_adaptive.get_embeddings()

chunks: list[Document] = []
for doc in docs:
    src = Path(doc.metadata.get("source", "")).name
    label, parts, _, _ = rag_adaptive.pick_best_strategy(doc.page_content, strategies)
    header = f"Document: {src}\nStrategy: {label}\nCategory: travel knowledge base\n---\n"
    for i, text in enumerate(parts):
        chunks.append(Document(
            page_content=header + text,
            metadata={"source": src, "doc_id": src, "chunk_index": i, "strategy": label},
        ))

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="travel_advanced",
)
print(f"Indexed {len(chunks)} contextual chunks to {CHROMA_DIR}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8463.63it/s]


Indexed 73 contextual chunks to c:\Users\Azooo\langchain-bootcamp\.chroma_advanced_rag


## Parent-child retrieval

**What:** Search small **child** chunks, return the larger **parent** section for the LLM to read.

**Problem it fixes:** Small chunks match queries precisely but lack surrounding context. Large chunks match poorly because the signal is diluted.

**How it works:** Store parents (big sections) and children (searchable snippets) in linked stores. Retrieval hits a child ID, then loads the parent text for generation.

**In this cell:** Query with a short phrase and inspect parent doc size vs what vector search alone would return.


In [17]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

parent_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=100)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
store = InMemoryStore()
pc_store = Chroma(
    collection_name="travel_parent_child",
    embedding_function=embeddings,
    persist_directory=str(ROOT / ".chroma_parent_child"),
)
parent_retriever = ParentDocumentRetriever(
    vectorstore=pc_store,
    docstore=store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)
parent_retriever.add_documents(docs)
parents = parent_retriever.invoke("Lisbon travel safety and entry requirements")
print(f"Parent docs returned: {len(parents)}")
for p in parents[:2]:
    print(" -", Path(p.metadata.get("source", "")).name, f"({len(p.page_content)} chars)")


Parent docs returned: 2
 - state_dept_portugal_travel.txt (1376 chars)
 - state_dept_portugal_travel.txt (1361 chars)


## Hybrid search (BM25 + vectors)

**What:** Run **keyword search** (BM25) and **semantic search** (embeddings) together, then merge the ranked lists.

**Problem it fixes:** Vectors miss exact tokens (regulation numbers, place names, "Schengen"). Keywords miss paraphrases and synonyms.

**How it works:** `BM25Retriever` finds literal term matches. The vector store finds meaning-level neighbors. `EnsembleRetriever` fuses both with reciprocal rank fusion (RRF).

**In this cell:** Compare hybrid top sources to what either method alone would surface.


In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

bm25 = BM25Retriever.from_documents(chunks)
bm25.k = 4
dense = vectorstore.as_retriever(search_kwargs={"k": 4})
hybrid = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5])
hybrid_docs = hybrid.invoke("Schengen visa 90 days travel rules")
print("Hybrid top sources:")
for i, d in enumerate(hybrid_docs[:4], 1):
    print(f" [{i}] {d.metadata.get('source')} chunk={d.metadata.get('chunk_index')}")


Hybrid top sources:
 [1] wikipedia_schengen_visa.txt chunk=11
 [2] wikipedia_schengen_visa.txt chunk=1
 [3] wikipedia_schengen_visa.txt chunk=10
 [4] wikipedia_schengen_visa.txt chunk=9


## MMR (diverse retrieval)

**What:** **Maximal Marginal Relevance** picks results that are relevant but not redundant.

**Problem it fixes:** Plain top-k vector search often returns five nearly identical chunks from the same Wikipedia section.

**How it works:** Fetch a larger candidate pool (`fetch_k`), then select `k` items that score high on relevance while penalizing similarity to chunks already chosen.

**In this cell:** MMR on a Lisbon query should spread results across more source files than vanilla similarity search.


In [ ]:
mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 12, "lambda_mult": 0.5},
)
mmr_docs = mmr.invoke("best season to visit Lisbon")
print(f"MMR returned {len(mmr_docs)} diverse chunks:")
for i, d in enumerate(mmr_docs, 1):
    print(f" [{i}] {d.metadata.get('source')}: {d.page_content[:]}...")


MMR returned 4 diverse chunks:
 [1] wikipedia_lisbon.txt: Document: wikipedia_lisbon.txt
Strategy: langch_recurs_1100
Category: ...
 [2] state_dept_portugal_travel.txt: Document: state_dept_portugal_travel.txt
Strategy: our_recurs_1100
Cat...
 [3] wikipedia_lisbon.txt: Document: wikipedia_lisbon.txt
Strategy: langch_recurs_1100
Category: ...
 [4] wikipedia_portugal.txt: Document: wikipedia_portugal.txt
Strategy: langch_recurs_1100
Category...


## LLM reranking

**What:** After vector search, ask the LLM to **score each candidate passage** for relevance to the query.

**Problem it fixes:** Embedding similarity is a rough proxy. The top vector hit can be from the right file but the wrong paragraph (or off-topic entirely).

**How it works:** Retrieve `k` chunks, strip indexing headers, score each passage 0 to 10 with the LLM, sort by score, use the winner for generation.

**In this cell:** Query EU delay compensation rights. Reranking should boost the passage that actually discusses compensation, not a unrelated section from the same document.


In [18]:
import re
from langchain_core.messages import HumanMessage

model = require_llm()
query = "EU air passenger delay compensation rights"
candidates = vectorstore.similarity_search(query, k=5)

def passage_body(doc):
    # Drop contextual indexing header; score the actual chunk text.
    return doc.page_content.split("---\n", 1)[-1].strip()[:400]

scores = []
for doc in candidates:
    prompt = (
        f"Query: {query}\n\n"
        f"Passage:\n{passage_body(doc)}\n\n"
        "Rate relevance from 0 (not relevant) to 10 (directly answers). "
        "Reply with one integer only."
    )
    raw = model.invoke([HumanMessage(content=prompt)]).content.strip()
    match = re.search(r"\d+", raw)
    scores.append(min(10, int(match.group())) if match else 0)

ranked = sorted(zip(scores, candidates), key=lambda pair: pair[0], reverse=True)
print("Reranked passages:")
for score, doc in ranked:
    print(f"  {score:2d}  {doc.metadata.get('source')}")
print("Top reranked source:", ranked[0][1].metadata.get("source"), "score=", ranked[0][0])


Reranked passages:
  10  eu_air_passenger_rights.txt
  10  eu_air_passenger_rights.txt
   0  eu_air_passenger_rights.txt
   0  eu_air_passenger_rights.txt
   0  wikipedia_schengen_visa.txt
Top reranked source: eu_air_passenger_rights.txt score= 10


## HyDE (Hypothetical Document Embeddings)

**What:** Have the LLM write a **hypothetical answer**, embed that text, and search the vector store with it.

**Problem it fixes:** Users ask in different words than the corpus uses. A short question may sit far from the right chunk in embedding space.

**How it works:** `query` to LLM draft answer to embed hypothetical doc to similarity search to real chunks. The fake answer bridges vocabulary between question and documents.

**In this cell:** TSA liquids rules. HyDE often beats searching with the raw question alone.

Requires a live LLM API key.


In [ ]:
from langchain_core.messages import HumanMessage

model = require_llm()
query = "Can I carry liquids in hand luggage through airport security?"
hyde_doc = model.invoke([
    HumanMessage(content=f"Write a short factual paragraph answering: {query}")
]).content
hyde_hits = vectorstore.similarity_search(hyde_doc, k=3)
print("HyDE retrieval sources:", [h.metadata.get("source") for h in hyde_hits])
print("Top snippet:", hyde_hits[0].page_content[:220], "...")


HyDE retrieval sources: ['tsa_what_can_i_bring.txt', 'tsa_what_can_i_bring.txt', 'wikipedia_airline_baggage.txt']
Top snippet: Document: tsa_what_can_i_bring.txt
Strategy: our_recurs_600
Category: travel knowledge base
---


 Alcoholic beverages with more than 24% but not more than 70% alcohol are limited in checked bags to 5 liters (1.3 gallons ...


## Corrective RAG (CRAG)

**What:** **Grade** whether retrieved chunks are good enough to answer. If not, **correct** by searching the web instead of forcing a bad answer.

**Problem it fixes:** Naive RAG always trusts local retrieval. Weak or empty matches lead to confident hallucinations.

**How it works:** Retrieve local chunks, LLM labels them relevant or irrelevant. If irrelevant, call Tavily web search and answer from fresh results. If relevant, answer from corpus.

**In this cell:** A query your travel corpus cannot answer triggers the web fallback path.

Requires `DEEPSEEK_API_KEY` and `TAVILY_API_KEY`.


In [ ]:
import os
from langchain_core.messages import HumanMessage

model = require_llm()
query = "quantum computing error correction codes 2026"
local = vectorstore.similarity_search(query, k=3)
grade_prompt = (
    f"Query: {query}\n\n"
    "Are these passages relevant enough to answer the query? "
    "Reply with exactly RELEVANT or IRRELEVANT.\n\n"
    + "\n---\n".join(d.page_content[:220] for d in local)
)
grade = model.invoke([HumanMessage(content=grade_prompt)]).content.strip().upper()
print("Retrieval grade:", grade)
if "IRRELEVANT" in grade:
    if not os.getenv("TAVILY_API_KEY"):
        raise RuntimeError("CRAG web fallback requires TAVILY_API_KEY in .env")
    from langchain_tavily import TavilySearch
    web = TavilySearch(max_results=3)
    print(web.invoke({"query": query}))
else:
    print("Local corpus sufficient:", local[0].page_content[:220])


Retrieval grade: IRRELEVANT
{'query': 'quantum computing error correction codes 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://thequantuminsider.com/2026/06/09/iqm-announces-novel-quantum-error-correction-approach-toward-fault-tolerant-quantum-computing', 'title': 'IQM Announces Novel Quantum Error Correction Approach Toward Fault-Tolerant Quantum Computing', 'content': '# IQM Announces Novel Quantum Error Correction Approach Toward Fault-Tolerant Quantum Computing. PRESS RELEASE — IQM Quantum Computers, the global leader in superconducting quantum computers, has developed a novel quantum error-correcting code that achieves up to three orders of magnitude lower logical error rates than the surface code, also requiring up to eight times fewer physical qubits. Unlike many alternative high-performance quantum error-correction approaches, the new code also maintains a comparatively low hardware complexity, marking a significant advancement to

## Agentic RAG (A-RAG)

**What:** Give the agent **retrieval tools** instead of a fixed retrieve-then-answer pipeline.

**Problem it fixes:** One retrieval strategy does not fit every question. Some need keywords, some need semantic search, some need the full chunk after a hit.

**How it works:** Expose `keyword_search`, `semantic_search`, and `read_chunk` as tools. The agent decides which to call, in what order, based on the user question. Watch `DATAFLOW` for the tool loop.

**In this cell:** The agent picks retrieval granularity like a human researcher would.

Requires a live LLM API key.


In [ ]:
from langchain_core.tools import tool

CHUNK_BY_ID = {
    f"{d.metadata['doc_id']}:{d.metadata['chunk_index']}": d for d in chunks
}

@tool
def keyword_search(keywords: str, k: int = 3) ->  str:
    """Exact keyword search over the travel corpus (comma-separated terms)."""
    terms = [t.strip().lower() for t in keywords.split(",") if t.strip()]
    scored = []
    for doc in chunks:
        text = doc.page_content.lower()
        score = sum(text.count(term) * len(term) for term in terms)
        if score:
            scored.append((score, doc))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    lines = []
    for score, doc in scored[:k]:
        cid = f"{doc.metadata['doc_id']}:{doc.metadata['chunk_index']}"
        lines.append(f"[{cid} score={score}]\n{doc.page_content[:250]}")
    return "\n---\n".join(lines) or "No keyword matches."

@tool
def semantic_search(query: str, k: int = 3) ->  str:
    """Semantic search over indexed travel knowledge."""
    hits = vectorstore.similarity_search(query, k=k)
    lines = []
    for doc in hits:
        cid = f"{doc.metadata.get('doc_id')}:{doc.metadata.get('chunk_index')}"
        lines.append(f"[{cid}]\n{doc.page_content[:250]}")
    return "\n---\n".join(lines)

@tool
def read_chunk(chunk_id: str) ->  str:
    """Read the full chunk text by id, e.g. wikipedia_lisbon.txt:2"""
    doc = CHUNK_BY_ID.get(chunk_id)
    if doc is None:
        return f"Unknown chunk_id: {chunk_id}"
    return doc.page_content

print("DATAFLOW (retrieval tools)")
print("1. keyword_search(Schengen,visa)")
print(preview(keyword_search.invoke({"keywords": "Schengen,visa", "k": 2})))
print("2. semantic_search(TSA liquids carry-on rules)")
print(preview(semantic_search.invoke({"query": "TSA liquids carry-on rules", "k": 2})))


DATAFLOW (retrieval tools)
1. keyword_search(Schengen,visa)
[wikipedia_schengen_visa.txt:2 score=240] Document: wikipedia_schengen_visa.txt Strategy: langch_recurs_1100 Category: travel knowledge base --- === Cyprus === Cyprus as EU member ...
2. semantic_search(TSA liquids carry-on rules)
[tsa_what_can_i_bring.txt:4] Document: tsa_what_can_i_bring.txt Strategy: our_recurs_600 Category: travel knowledge base ---    Alcoholic beverages with more than 24% but not more ...
